# 03 — Train YOLOv8s Detector-only

Notebook này train/evaluate YOLOv8s trên YOLO 7-class. Đã có fixed-size guard để tránh lỗi ảnh 640 và 1280 trong cùng batch.


# Common setup notes

Trước khi chạy notebook:

1. Bật GPU trong Kaggle: `Settings → Accelerator → GPU`.
2. Thêm dataset TACO/Roboflow ở mục `Add Input`.
3. Nếu dùng W&B online, thêm Kaggle Secret:
   - Name: `WANDB_API_KEY`
   - Value: API key của tài khoản W&B.
4. Sửa biến `REPO_URL`, `WANDB_ENTITY`, `TACO_ROOT` hoặc `ROBOFLOW_ROOT` cho đúng với nhóm bạn.

Các notebook này dùng nguyên command-line scripts trong GitHub repo đã hoàn thiện, không viết lại training logic trong notebook.


In [ ]:
import os
from pathlib import Path

# ===== EDIT ME =====
REPO_URL = "https://github.com/nnnhuan251-hcmus/Detect-Waste-ADN.git"
REPO_DIR = Path("/kaggle/working/Detect-Waste-ADN")
BRANCH = "main"

# W&B
USE_WANDB = False              # đổi True khi smoke W&B đã pass
WANDB_MODE = "online"          # "online" hoặc "offline"
WANDB_ENTITY = ""              # ví dụ: "waste-detection-team"; để "" nếu dùng default entity

# Kaggle data config
DATA_CONFIG = "configs/data/taco_7class.yaml"

# Runtime flags
RUN_SMOKE = True
RUN_FULL = False               # chỉ đổi True sau khi smoke pass
RUN_EVAL = True
RUN_WANDB_EVAL_UPLOAD = False

os.environ["REPO_URL"] = REPO_URL
os.environ["REPO_DIR"] = str(REPO_DIR)
os.environ["BRANCH"] = BRANCH
os.environ["USE_WANDB"] = "1" if USE_WANDB else "0"
os.environ["WANDB_MODE"] = WANDB_MODE
os.environ["WANDB_ENTITY"] = WANDB_ENTITY
os.environ["DATA_CONFIG"] = DATA_CONFIG
os.environ["RUN_SMOKE"] = "1" if RUN_SMOKE else "0"
os.environ["RUN_FULL"] = "1" if RUN_FULL else "0"
os.environ["RUN_EVAL"] = "1" if RUN_EVAL else "0"
os.environ["RUN_WANDB_EVAL_UPLOAD"] = "1" if RUN_WANDB_EVAL_UPLOAD else "0"

print("REPO_DIR:", REPO_DIR)
print("DATA_CONFIG:", DATA_CONFIG)
print("USE_WANDB:", USE_WANDB, "| WANDB_MODE:", WANDB_MODE, "| WANDB_ENTITY:", WANDB_ENTITY or "<default>")
print("RUN_SMOKE:", RUN_SMOKE, "| RUN_FULL:", RUN_FULL, "| RUN_EVAL:", RUN_EVAL)


In [ ]:
%%bash
set -e

cd /kaggle/working

if [ ! -d "$REPO_DIR/.git" ]; then
  echo "Cloning repo..."
  git clone --branch "$BRANCH" "$REPO_URL" "$REPO_DIR"
else
  echo "Repo exists. Pulling latest changes..."
  cd "$REPO_DIR"
  git fetch origin "$BRANCH"
  git checkout "$BRANCH"
  git pull origin "$BRANCH"
fi

cd "$REPO_DIR"

python -m pip install -q --upgrade pip
pip install -q -r requirements.txt

if [ -f requirements-optional.txt ]; then
  pip install -q -r requirements-optional.txt || true
fi

export PYTHONPATH="$REPO_DIR/src:$PYTHONPATH"
echo "PYTHONPATH=$PYTHONPATH"


In [ ]:
import os

if os.environ.get("USE_WANDB") == "1":
    import wandb
    try:
        from kaggle_secrets import UserSecretsClient
        key = UserSecretsClient().get_secret("WANDB_API_KEY")
        os.environ["WANDB_API_KEY"] = key
        wandb.login()
        print("W&B login passed.")
    except Exception as exc:
        print("W&B login failed:", repr(exc))
        print("Switching to offline mode for safety.")
        os.environ["WANDB_MODE"] = "offline"
else:
    print("W&B disabled for this notebook.")


In [ ]:
%%bash
set -e

cd "$REPO_DIR"
export PYTHONPATH="$REPO_DIR/src:$PYTHONPATH"

echo "=== Compile check ==="
python -m compileall -q src scripts

echo "=== Core import check ==="
python - <<'PY'
import torch
from pathlib import Path

from waste_detection.config.config_loader import ConfigLoader
from waste_detection.data.coco_reader import CocoReader
from waste_detection.data.coco_validator import COCOValidator
from waste_detection.data.crop_dataset import CropClassificationDataset
from waste_detection.models.efficientnet_classifier import EfficientNetB0Classifier
from waste_detection.training.detector_trainer import DetectorTrainer
from waste_detection.training.classifier_trainer import ClassifierTrainer
from waste_detection.inference.hybrid_predictor import HybridPredictor
from waste_detection.evaluation.classification_metrics import ClassificationMetrics
from waste_detection.evaluation.hybrid_metrics import HybridMetrics

print("torch:", torch.__version__)
print("cuda_available:", torch.cuda.is_available())
print("core imports passed")
PY


In [ ]:
%%bash
set -e

cd "$REPO_DIR"
export PYTHONPATH="$REPO_DIR/src:$PYTHONPATH"

echo "=== Detector fixed-size guard ==="
python - <<'PY'
from pathlib import Path
import re

p = Path("src/waste_detection/training/detector_trainer.py")
text = p.read_text(encoding="utf-8")

checks = {
    "imgsz": bool(re.search(r'["\']imgsz["\']\s*:', text)),
    "rect_false": bool(re.search(r'["\']rect["\']\s*:\s*False', text)),
    "multi_scale_false": bool(re.search(r'["\']multi_scale["\']\s*:\s*False', text)),
}

print(checks)

missing = [name for name, ok in checks.items() if not ok]
if missing:
    raise SystemExit(
        "DetectorTrainer chưa khóa fixed image size đủ an toàn. "
        f"Thiếu: {missing}. "
        "Hãy sửa _build_train_args(): đặt imgsz=int(detector.input_size), rect=False, multi_scale=False. "
        "Nếu không sửa, YOLO/RT-DETR có thể nhận ảnh 640 và 1280 trong cùng batch và crash."
    )

print("Detector fixed-size guard passed.")
PY


In [ ]:
%%bash
set -e

cd "$REPO_DIR"
export PYTHONPATH="$REPO_DIR/src:$PYTHONPATH"

python - <<'PY'
from pathlib import Path
import yaml
import copy

src_dir = Path("configs/experiments")
out_dir = src_dir / "smoke"
out_dir.mkdir(parents=True, exist_ok=True)

def load_yaml(path):
    with open(path, "r", encoding="utf-8") as f:
        return yaml.safe_load(f)

def save_yaml(path, data):
    with open(path, "w", encoding="utf-8") as f:
        yaml.safe_dump(data, f, sort_keys=False, allow_unicode=True)

def make_smoke(src_name, out_name, epochs=1, batch_size=2, num_workers=2, patience=0):
    cfg = load_yaml(src_dir / src_name)
    cfg["experiment"]["name"] = out_name.replace(".yaml", "")
    cfg["experiment"]["description"] = "SMOKE TEST - " + str(cfg["experiment"].get("description", ""))

    cfg.setdefault("training", {})
    cfg["training"]["epochs"] = epochs
    cfg["training"]["batch_size"] = batch_size
    cfg["training"]["num_workers"] = num_workers
    cfg["training"]["patience"] = patience
    cfg["training"]["early_stopping"] = False
    cfg["training"]["mixed_precision"] = True

    if cfg.get("scheduler", {}).get("name") == "cosine_annealing":
        cfg["training"]["epochs"] = max(2, epochs)
        cfg["scheduler"].setdefault("warmup", {})
        cfg["scheduler"]["warmup"]["warmup_epochs"] = 1

    cfg.setdefault("tracking", {})
    cfg["tracking"]["use_wandb"] = False
    cfg["tracking"]["mode"] = "disabled"
    cfg["tracking"]["log_model"] = False
    cfg["tracking"]["watch_model"] = False
    cfg["tracking"]["log_code"] = False

    save_yaml(out_dir / out_name, cfg)

make_smoke("run1_baseline.yaml", "smoke_run1_baseline.yaml", epochs=1, batch_size=2)
make_smoke("run2_strong_aug.yaml", "smoke_run2_strong_aug.yaml", epochs=1, batch_size=2)
make_smoke("run3_freeze_cosine_warmup.yaml", "smoke_run3_freeze_cosine_warmup.yaml", epochs=2, batch_size=2)
make_smoke("run4_balanced_sampler.yaml", "smoke_run4_balanced_sampler.yaml", epochs=1, batch_size=2)

# RT-DETR smoke: force tiny batch
cfg = load_yaml(src_dir / "run1_baseline.yaml")
cfg["experiment"]["name"] = "smoke_run1_rtdetr"
cfg["experiment"]["description"] = "SMOKE TEST - RT-DETR-L tiny batch."
cfg.setdefault("training", {})
cfg["training"]["epochs"] = 1
cfg["training"]["batch_size"] = 1
cfg["training"]["num_workers"] = 2
cfg["training"]["patience"] = 0
cfg["training"]["early_stopping"] = False
cfg["training"]["mixed_precision"] = True
cfg.setdefault("tracking", {})
cfg["tracking"]["use_wandb"] = False
cfg["tracking"]["mode"] = "disabled"
cfg["tracking"]["log_model"] = False
cfg["tracking"]["watch_model"] = False
cfg["tracking"]["log_code"] = False
save_yaml(out_dir / "smoke_run1_rtdetr.yaml", cfg)

print("Smoke configs:")
for p in sorted(out_dir.glob("*.yaml")):
    print(" -", p)
PY


## 1. Kiểm tra YOLO 7-class data


In [ ]:
%%bash
set -e
cd "$REPO_DIR"

# Tự động tìm và giải nén data từ Notebook 01 nếu đã được Add Input
if [ ! -d "data/processed/yolo_7class" ]; then
  echo "Thử tìm và giải nén dữ liệu từ /kaggle/input/..."
  TAR_FILE=$(find /kaggle/input -name "preprocessed_data_and_reports.tar.gz" | head -n 1)
  if [ -n "$TAR_FILE" ]; then
    echo "Tìm thấy: $TAR_FILE"
    tar -xzf "$TAR_FILE" -C .
    echo "Giải nén thành công!"
  else
    echo "❌ KHÔNG tìm thấy preprocessed_data_and_reports.tar.gz."
    echo "👉 Bạn vui lòng nhìn sang menu phải, bấm 'Add Input' -> 'Your Work' -> Chọn Notebook 01 trước nhé."
    exit 1
  fi
fi

test -f data/processed/yolo_7class/data.yaml || (echo "Missing yolo_7class/data.yaml. Chạy notebook 01 trước." && exit 1)
cat data/processed/yolo_7class/data.yaml


## 2. Smoke train YOLOv8s run1/run2/run3


In [ ]:
%%bash
set -e

cd "$REPO_DIR"
export PYTHONPATH="$REPO_DIR/src:$PYTHONPATH"

if [ "$RUN_SMOKE" != "1" ]; then
  echo "RUN_SMOKE=0 → skip."
  exit 0
fi

for RUN in smoke_run1_baseline smoke_run2_strong_aug smoke_run3_freeze_cosine_warmup
do
  echo "=== Smoke YOLOv8s: $RUN ==="
  python scripts/train/train_detector.py \
    --data-config "$DATA_CONFIG" \
    --model-config configs/models/yolov8s.yaml \
    --experiment-config configs/experiments/smoke/${RUN}.yaml
done


## 3. Train thật YOLOv8s run1/run2/run3


In [ ]:
%%bash
set -e

cd "$REPO_DIR"
export PYTHONPATH="$REPO_DIR/src:$PYTHONPATH"


WANDB_FLAGS=""
if [ "$USE_WANDB" = "1" ]; then
  WANDB_FLAGS="--use-wandb --wandb-mode $WANDB_MODE"
  if [ -n "$WANDB_ENTITY" ]; then
    WANDB_FLAGS="$WANDB_FLAGS --wandb-entity $WANDB_ENTITY"
  fi
fi
echo "WANDB_FLAGS=$WANDB_FLAGS"


if [ "$RUN_FULL" != "1" ]; then
  echo "RUN_FULL=0 → skip full YOLOv8s training."
  exit 0
fi

for RUN in run1_baseline run2_strong_aug run3_freeze_cosine_warmup
do
  echo "=== Train YOLOv8s: $RUN ==="
  python scripts/train/train_detector.py \
    --data-config "$DATA_CONFIG" \
    --model-config configs/models/yolov8s.yaml \
    --experiment-config configs/experiments/${RUN}.yaml \
    $WANDB_FLAGS
done


## 4. Evaluate YOLOv8s


In [ ]:
%%bash
set -e

cd "$REPO_DIR"
export PYTHONPATH="$REPO_DIR/src:$PYTHONPATH"

if [ "$RUN_EVAL" != "1" ]; then
  echo "RUN_EVAL=0 → skip."
  exit 0
fi

for RUN in run1_baseline run2_strong_aug run3_freeze_cosine_warmup
do
  CKPT="outputs/checkpoints/yolov8s/${RUN}/weights/best.pt"
  if [ ! -f "$CKPT" ]; then
    echo "Skip YOLOv8s eval $RUN: missing $CKPT"
    continue
  fi

  python scripts/eval/evaluate_detector.py \
    --data-config "$DATA_CONFIG" \
    --model-config configs/models/yolov8s.yaml \
    --experiment-config configs/experiments/${RUN}.yaml \
    --weights "$CKPT" \
    --split test
done


## 5. Tổng hợp YOLOv8s training/eval artifacts


In [ ]:
from pathlib import Path
import pandas as pd
import json
import matplotlib.pyplot as plt

Path("outputs/metrics/final").mkdir(parents=True, exist_ok=True)
Path("outputs/figures/final").mkdir(parents=True, exist_ok=True)

runs = ["run1_baseline", "run2_strong_aug", "run3_freeze_cosine_warmup"]
rows = []

for run in runs:
    results_csv = Path(f"outputs/checkpoints/yolov8s/{run}/results.csv")
    row = {"model": "yolov8s", "run": run}
    if results_csv.exists():
        df = pd.read_csv(results_csv)
        df.columns = [c.strip() for c in df.columns]
        last = df.iloc[-1].to_dict()
        for key in df.columns:
            low = key.lower()
            if "map50-95" in low or "map50_95" in low:
                row["val_map50_95_last"] = last.get(key)
            elif "map50" in low and "map50-95" not in low:
                row["val_map50_last"] = last.get(key)
            elif "precision" in low:
                row["val_precision_last"] = last.get(key)
            elif "recall" in low:
                row["val_recall_last"] = last.get(key)
        row["results_csv"] = str(results_csv)
    else:
        row["missing_results_csv"] = True

    # Nếu evaluate_detector.py đã lưu metrics.json, tự động lấy thêm.
    for p in Path("outputs/metrics/detector").rglob("metrics.json"):
        if "yolov8s" in str(p) and run in str(p):
            try:
                m = json.loads(p.read_text(encoding="utf-8"))
                row.update({f"test_{k}": v for k, v in m.items() if isinstance(v, (int, float, str)) or v is None})
            except Exception:
                pass

    rows.append(row)

summary = pd.DataFrame(rows)
summary.to_csv("outputs/metrics/final/yolov8s_summary.csv", index=False)
display(summary)

if "val_map50_last" in summary.columns and summary["val_map50_last"].notna().any():
    plt.figure(figsize=(9, 5))
    plt.bar(summary["run"], summary["val_map50_last"])
    plt.xticks(rotation=25, ha="right")
    plt.ylabel("Validation mAP50")
    plt.title("YOLOv8s validation mAP50 by run")
    plt.tight_layout()
    plt.savefig("outputs/figures/final/yolov8s_map50_comparison.png", dpi=300)
    plt.show()


## 6. Đóng gói outputs YOLOv8s


In [ ]:
%%bash
set -e

cd "$REPO_DIR"
mkdir -p /kaggle/working/artifacts

tar -czf /kaggle/working/artifacts/yolov8s_metrics_figures_logs.tar.gz \
  outputs/metrics \
  outputs/figures \
  outputs/logs \
  outputs/checkpoints/yolov8s/*/*.png \
  outputs/checkpoints/yolov8s/*/results.csv \
  2>/dev/null || true

tar -czf /kaggle/working/artifacts/yolov8s_checkpoints.tar.gz \
  outputs/checkpoints/yolov8s \
  2>/dev/null || true

ls -lh /kaggle/working/artifacts
